# Sanity checks and actual-engine case stories

Boundary identities and selected patient/clinic stories make assumptions inspectable. Every story below is captured during an ordinary `BranchedSimulationEngine.run` through `TracingScenario`; there is no handwritten engine loop. A fresh unobserved run is compared exactly with each traced run. Both factual and counterfactual measurements are included.

These small public-default runs are **illustrations**, not evidence for an institutional experiment. Run the same tracing wrapper with the reviewed configuration and seed to produce experiment-specific evidence. Examples establish mechanics, not clinical validity or population-level effectiveness. Selection rules and candidate counts are stated with each story. JSON trace references resolve under `reports/sanity_traces/` after execution.

In [ ]:
from pathlib import Path
import sys
import json
from dataclasses import asdict

# Works from a fresh kernel launched at either repository root or examples/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'healthcare_sim_sdk').is_dir())
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from healthcare_sim_sdk.ml.performance import theoretical_ppv, theoretical_ppv_bounds
from healthcare_sim_sdk.core.engine import BranchedSimulationEngine, CounterfactualMode
from healthcare_sim_sdk.core.scenario import TimeConfig
from healthcare_sim_sdk.core.rng import RNGPartitioner
from healthcare_sim_sdk.experiments.trace import TracingScenario
from healthcare_sim_sdk.scenarios.noshow_overbooking.scenario import ClinicConfig, NoShowOverbookingScenario
from healthcare_sim_sdk.scenarios.stroke_prevention.scenario import (
    StrokeConfig, StrokePreventionScenario, BASE_RISKS, AR1_MODS, TX_EFFECT, CURRENT_RISKS,
)
TRACE_DIR = ROOT / 'reports' / 'sanity_traces'
TRACE_DIR.mkdir(parents=True, exist_ok=True)

def reconcile(traced, plain):
    for branch in ('outcomes', 'counterfactual_outcomes'):
        for t, out in getattr(traced, branch).items():
            reference = getattr(plain, branch)[t]
            np.testing.assert_array_equal(out.entity_ids, reference.entity_ids)
            np.testing.assert_array_equal(out.events, reference.events)
            for key, values in out.secondary.items():
                np.testing.assert_array_equal(values, reference.secondary[key])
    for t, prediction in traced.predictions.items():
        np.testing.assert_array_equal(prediction.scores, plain.predictions[t].scores)
        np.testing.assert_array_equal(traced.interventions[t].treated_indices,
                                      plain.interventions[t].treated_indices)

def persist_trace(name, wrapper, selection, parameters):
    path = TRACE_DIR / f'{name}.json'
    path.write_text(json.dumps({'purpose': 'public-default illustration',
                               'selection': selection, 'parameters': parameters,
                               'records': wrapper.to_rows()}, indent=2))
    print(f'Trace artifact: {path}')
    return path

---

# Part 1: Boundary Condition Sanity Checks

## 1.1 Prevalence vs PPV — The Bayes' Theorem Reality Check

The single most important chart for anyone evaluating an ML model in healthcare. At fixed sensitivity and specificity, Bayes' theorem determines PPV. There is no prevalence-only ceiling: perfect specificity can achieve PPV of one.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PPV bounds at different prevalences
for prev, color in [(0.01, 'red'), (0.05, 'orange'), (0.13, 'blue'), (0.30, 'green'), (0.50, 'purple')]:
    sens_range, bounds = theoretical_ppv_bounds(prev, specificities=[0.90])
    axes[0].plot(sens_range, bounds["spec_0.90"], label=f'prev={prev:.0%}', color=color, linewidth=2)
axes[0].set_xlabel('Sensitivity')
axes[0].set_ylabel('PPV at 90% specificity')
axes[0].set_title('PPV at 90% Specificity\n(Bayes\' Theorem)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1)

# Right: Verify known identities
checks = []
for v in [0.70, 0.80, 0.90, 0.95]:
    ppv = theoretical_ppv(0.50, v, v)
    checks.append((f'50% prev, sens=spec={v:.2f}', v, ppv, abs(ppv - v) < 0.001))

for prev in [0.01, 0.05, 0.13]:
    ppv = theoretical_ppv(prev, 0.80, 1.0)
    checks.append((f'prev={prev:.0%}, spec=1.0', 1.0, ppv, abs(ppv - 1.0) < 0.001))

for prev in [0.05, 0.13, 0.50]:
    ppv = theoretical_ppv(prev, 1.0, 0.0)
    checks.append((f'prev={prev:.0%}, spec=0.0', prev, ppv, abs(ppv - prev) < 0.001))

ppv = theoretical_ppv(0.01, 0.80, 0.99)
checks.append(('1% prev, 99% spec', 0.447, ppv, abs(ppv - 0.447) < 0.01))

colors = ['green' if ok else 'red' for _, _, _, ok in checks]
y_pos = range(len(checks))
axes[1].barh(y_pos, [actual for _, _, actual, _ in checks], color=colors, alpha=0.7, label='Actual')
axes[1].scatter([expected for _, expected, _, _ in checks], y_pos, 
                color='black', marker='|', s=200, linewidths=2, zorder=5, label='Expected')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels([name for name, _, _, _ in checks], fontsize=8)
axes[1].set_xlabel('PPV')
axes[1].set_title('Bayes\' Theorem Identity Checks\n(green=pass, black marker=expected)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
print(f"All {sum(1 for _,_,_,ok in checks if ok)}/{len(checks)} identity checks passed.")
assert all(ok for _, _, _, ok in checks), "Bayes identity check failed"

## 1.2 No-Show Scenario at Boundary Conditions

Four boundary tests where the answer is known before running:

| Condition | Expected outcome |
|-----------|-----------------|
| Threshold = 1.0 | Zero overbookings (no predicted prob reaches 1.0) |
| Max individual overbooks = 0 | Zero overbookings (no eligible candidates) |
| Threshold = 1.0 + BRANCHED | Factual == counterfactual (no interventions = identical branches) |
| Low threshold vs high | More overbookings at lower threshold |

In [ ]:
def run_noshow(n_days=15, n_patients=500, seed=42, **kwargs):
    tc = TimeConfig(n_timesteps=n_days, timestep_duration=1/365,
                    timestep_unit="day", prediction_schedule=list(range(n_days)))
    cc = ClinicConfig(n_providers=4, slots_per_provider_per_day=10, max_overbook_per_provider=2)
    defaults = dict(n_patients=n_patients, base_noshow_rate=0.13, model_type="predictor",
                    model_auc=0.83, overbooking_threshold=0.30, clinic_config=cc)
    defaults.update(kwargs)
    sc = NoShowOverbookingScenario(time_config=tc, seed=seed, **defaults)
    return BranchedSimulationEngine(sc, CounterfactualMode.BRANCHED).run(n_patients)

# Test 1: Threshold = 1.0 → zero overbookings
r1 = run_noshow(overbooking_threshold=1.0)
ob1 = r1.outcomes[14].metadata["total_overbooked"]

# Test 2: Max individual = 0 → zero overbookings  
r2 = run_noshow(overbooking_threshold=0.01, max_individual_overbooks=0)
ob2 = r2.outcomes[14].metadata["total_overbooked"]

# Test 3: No intervention → factual == counterfactual
r3 = run_noshow(overbooking_threshold=1.0, n_days=10)
diffs = [np.abs(r3.outcomes[t].events - r3.counterfactual_outcomes[t].events).sum() 
         for t in range(1, 10)]

# Test 4: Monotonicity — lower threshold → more overbooking
r4_low = run_noshow(overbooking_threshold=0.10, n_days=20)
r4_high = run_noshow(overbooking_threshold=0.50, n_days=20)
ob_low = r4_low.outcomes[19].metadata["total_overbooked"]
ob_high = r4_high.outcomes[19].metadata["total_overbooked"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Test 1
ax = axes[0, 0]
ax.bar(['threshold=1.0', 'expected'], [ob1, 0], color=['blue', 'green'], alpha=0.7)
ax.set_title(f'Threshold=1.0 → Overbookings\nActual: {ob1} (expected: 0)')
ax.set_ylabel('Total overbookings')
ax.annotate('PASS' if ob1 == 0 else 'FAIL', xy=(0.5, 0.5), fontsize=20,
           color='green' if ob1 == 0 else 'red', ha='center', transform=ax.transAxes)

# Test 2  
ax = axes[0, 1]
ax.bar(['max_cap=0', 'expected'], [ob2, 0], color=['blue', 'green'], alpha=0.7)
ax.set_title(f'Max Cap=0 → Overbookings\nActual: {ob2} (expected: 0)')
ax.set_ylabel('Total overbookings')
ax.annotate('PASS' if ob2 == 0 else 'FAIL', xy=(0.5, 0.5), fontsize=20,
           color='green' if ob2 == 0 else 'red', ha='center', transform=ax.transAxes)

# Test 3
ax = axes[1, 0]
ax.bar(range(1, 10), diffs, color=['green' if d == 0 else 'red' for d in diffs])
ax.set_xlabel('Timestep')
ax.set_ylabel('|Factual - CF| total')
ax.set_title(f'No Intervention → F == CF\nMax diff: {max(diffs):.0f} (expected: 0)')
all_zero = all(d == 0 for d in diffs)
ax.annotate('PASS' if all_zero else 'FAIL', xy=(0.5, 0.5), fontsize=20,
           color='green' if all_zero else 'red', ha='center', transform=ax.transAxes)

# Test 4
ax = axes[1, 1]
ax.bar(['thresh=0.10', 'thresh=0.50'], [ob_low, ob_high], color=['blue', 'orange'], alpha=0.7)
ax.set_title(f'Lower Threshold → More Overbooking\n{ob_low} vs {ob_high}')
ax.set_ylabel('Total overbookings')
ok = ob_low >= ob_high
ax.annotate('PASS' if ok else 'FAIL', xy=(0.5, 0.5), fontsize=20,
           color='green' if ok else 'red', ha='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()
assert ob1 == ob2 == 0
assert all_zero
assert ob_low >= ob_high

# Part 2: Actual-engine trace-through

## 2.1 No-show: selected patients, both branches

Selection is made from the initial population using an independent reviewer RNG (seed 20260905): three uniformly sampled patients plus the patient with the highest baseline no-show risk. The high-risk case is explicitly mechanism-selected and does not estimate prevalence. Candidate count is 200; categories may overlap. The projector records only these patients and the clinic's small current/resolved worklists.

Each measurement row is linked by `trace_id` to `noshow.json`. Counts are cumulative; an unscheduled patient need not change that day. Day zero records no resolved appointments: its event array is empty and its time-series value is unobserved (NaN). The initial schedule is available for intervention on day zero and resolved on day one; each appointment ID is measured once. Run one additional timestep to observe outcomes for the final scheduled day.

In [ ]:
n_days = 20
reviewer_rng = RNGPartitioner(20260905).create_streams().population
selection = {}

def make_noshow():
    tc = TimeConfig(n_days, 1/365, 'day', list(range(n_days)))
    cc = ClinicConfig(n_providers=3, slots_per_provider_per_day=8, max_overbook_per_provider=2)
    return NoShowOverbookingScenario(time_config=tc, seed=42, n_patients=200,
            base_noshow_rate=0.13, overbooking_threshold=0.25,
            max_individual_overbooks=5, model_type='predictor', model_auc=0.83, clinic_config=cc)

def noshow_project(state):
    if not selection:
        candidates = sorted(state.patients)
        random_ids = reviewer_rng.choice(candidates, 3, replace=False).tolist()
        highest = max(candidates, key=lambda pid: state.patients[pid].base_noshow_probability)
        selection.update(candidate_count=len(candidates), random_seed=20260905,
                         random_ids=random_ids, highest_baseline_risk=highest,
                         unavailable_categories=[])
    selected = sorted(set(selection['random_ids'] + [selection['highest_baseline_risk']]))
    return {'patients': {str(pid): asdict(state.patients[pid]) for pid in selected},
            'schedule': [asdict(s) for s in state.schedule],
            'resolved_slots': [asdict(s) for s in state.resolved_slots]}

noshow_trace = TracingScenario(make_noshow(), noshow_project,
    stages=('population', 'measure_after', 'prediction', 'intervention', 'outcome'))
ns_result = BranchedSimulationEngine(noshow_trace).run(200)
reconcile(ns_result, BranchedSimulationEngine(make_noshow()).run(200))
ns_rows = noshow_trace.to_rows()
assert not ns_result.to_analysis().to_time_series()['observed'][0]
assert len(ns_result.outcomes[0].events) == 0
ns_path = persist_trace('noshow', noshow_trace, selection,
    {'seed': 42, 'n_patients': 200, 'n_days': n_days, 'overbooking_threshold': .25,
     'model_type': 'predictor', 'model_auc': .83, 'base_noshow_rate': .13,
     'max_individual_overbooks': 5, 'clinic': asdict(make_noshow().clinic_config)})
print('Selection:', selection)
print('Exact equality with the unobserved engine run: PASS')
for row in ns_rows:
    if row['stage'] != 'measure_after':
        continue
    for pid, patient in row['facts']['patients'].items():
        print(row['trace_id'], row['timestep'], row['branch'], 'patient', pid,
              'appointments', patient['n_past_appointments'],
              'no-shows', patient['n_past_noshows'], 'overbooked', patient['n_times_overbooked'])

Check that cumulative patient histories are nondecreasing within each branch. Counterfactual overbooking stays zero. Factual and counterfactual histories can diverge, so compare each against its own prior row.

## 2.2 Clinic day four: resolution observed on day five

This is a mechanism-selected day (day four, chosen before looking at outcomes), with all 24 appointment slots shown. The same recorded `measure_after` state produces the exported outcome. Slot IDs, rather than array positions, join assignment and resolution. No second manual simulation is used.

In [ ]:
clinic_row = next(row for row in ns_rows if row['stage'] == 'measure_after'
                  and row['timestep'] == 5 and row['branch'] == 'factual')
slots = clinic_row['facts']['resolved_slots']
assert slots, 'No resolved slots available for selected day'
print('Source:', ns_path.name, clinic_row['trace_id'], '| candidates:', len(slots))
print('slot | patient | score | overbooked | original showed | extra showed | collision | wait')
for slot in slots:
    collision = slot['is_overbooked'] and slot['original_showed'] and slot['overbooked_showed']
    print(slot['slot_id'], slot['patient_id'], round(slot['predicted_noshow_prob'], 3),
          slot['is_overbooked'], slot['original_showed'], slot['overbooked_showed'],
          collision, slot['wait_time_minutes'])
np.testing.assert_array_equal([s['slot_id'] for s in slots], ns_result.outcomes[5].entity_ids)
np.testing.assert_array_equal([0.0 if s['original_showed'] else 1.0 for s in slots],
                              ns_result.outcomes[5].events)
np.testing.assert_array_equal([s['wait_time_minutes'] for s in slots],
                              ns_result.outcomes[5].secondary['wait_times'])
print('Trace-to-outcome reconciliation: PASS')

Review whether overbooked slots pass the configured score threshold and whether both attendees explain collisions. A selected day is too small to establish a population no-show rate or score performance. Scores describe the slot when scheduled; today's predictions may belong to different slots than today's measured outcomes.

## 2.3 Both-branch cumulative clinic outcomes

These aggregates come directly from the same traced run. Inspect burden and collision growth together; neither alone establishes whether overbooking is worthwhile.

In [ ]:
days = sorted(ns_result.outcomes)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for branch, label in [('outcomes', 'Factual'), ('counterfactual_outcomes', 'Counterfactual')]:
    data = getattr(ns_result, branch)
    axes[0].plot(days, [data[t].metadata['mean_overbooking_burden'] for t in days], label=label)
    axes[1].plot(days, [data[t].metadata['total_collisions'] for t in days], label=label)
axes[0].set(title='Mean cumulative overbooking burden', xlabel='Day', ylabel='Overbookings per patient')
axes[1].set(title='Cumulative collisions', xlabel='Day', ylabel='Collisions')
for ax in axes:
    ax.legend()
    ax.grid(alpha=.3)
plt.tight_layout()
plt.show()
assert all(o.metadata['mean_overbooking_burden'] == 0 for o in ns_result.counterfactual_outcomes.values())

## 2.4 Stroke: paired risk trajectories and treatment timing

Three patients are uniformly sampled from 500 using the independent reviewer seed 20260906. A fourth case is selected as the 50th-highest initial risk, a stated mechanism-based choice. The plot shows that selected case; the trace includes all selected cases, including any without treatment/events. Unavailable treated/event categories are listed explicitly.

Annual risk is a latent scenario quantity. These synthetic paired patient outcomes expose mechanism behavior; individual counterfactuals are not observable in deployment.

In [ ]:
stroke_selection = {}
stroke_reviewer_rng = RNGPartitioner(20260906).create_streams().population
config = StrokeConfig(n_patients=500, n_weeks=52, prediction_interval=4,
                      intervention_effectiveness=.50)

def make_stroke():
    return StrokePreventionScenario(config=config, seed=42)

def stroke_project(state):
    if not stroke_selection:
        random_ids = stroke_reviewer_rng.choice(500, 3, replace=False).tolist()
        mechanism_id = int(np.argsort(state[BASE_RISKS])[-50])
        stroke_selection.update(candidate_count=500, random_seed=20260906,
            random_ids=random_ids, fiftieth_highest_initial_risk=mechanism_id)
    selected = sorted(set(stroke_selection['random_ids'] +
                          [stroke_selection['fiftieth_highest_initial_risk']]))
    return {'patients': {str(i): {'base_risk': state[BASE_RISKS, i],
                'ar1_mod': state[AR1_MODS, i], 'tx_effect': state[TX_EFFECT, i],
                'current_risk': state[CURRENT_RISKS, i]} for i in selected}}

stroke_trace = TracingScenario(make_stroke(), stroke_project,
    stages=('population', 'intervene_before', 'intervene_after', 'measure_after',
            'prediction', 'intervention', 'outcome'))
stroke_result = BranchedSimulationEngine(stroke_trace).run(500)
reconcile(stroke_result, BranchedSimulationEngine(make_stroke()).run(500))
srows = stroke_trace.to_rows()
selected_ids = sorted(set(stroke_selection['random_ids'] +
                         [stroke_selection['fiftieth_highest_initial_risk']]))
stroke_selection['unavailable_categories'] = []
if not any(i in intv.treated_indices for intv in stroke_result.interventions.values() for i in selected_ids):
    stroke_selection['unavailable_categories'].append('treated selected patient')
if not any(out.events[i] for out in stroke_result.outcomes.values() for i in selected_ids):
    stroke_selection['unavailable_categories'].append('factual event among selected patients')
stroke_path = persist_trace('stroke', stroke_trace, stroke_selection,
                           {'seed': 42, 'config': asdict(config)})
print('Selection:', stroke_selection)
pid = stroke_selection['fiftieth_highest_initial_risk']
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for branch in ('factual', 'counterfactual'):
    rows = [r for r in srows if r['stage'] == 'measure_after' and r['branch'] == branch]
    patient_rows = [r['facts']['patients'][str(pid)] for r in rows]
    for row, patient in zip(rows, patient_rows):
        np.testing.assert_allclose(patient['current_risk'],
            np.clip(patient['base_risk'] * patient['ar1_mod'] * patient['tx_effect'], 0, 1))
    axes[0].plot([r['timestep'] for r in rows], [p['current_risk'] for p in patient_rows], label=branch)
    axes[1].plot([r['timestep'] for r in rows], [p['tx_effect'] for p in patient_rows], label=branch)
axes[0].set(ylabel='Annual stroke risk', title=f'Patient {pid}: paired latent risk')
axes[1].set(xlabel='Week', ylabel='Treatment multiplier')
for ax in axes:
    ax.legend()
    ax.grid(alpha=.3)
plt.tight_layout()
plt.show()
print('Treatment and event source rows:')
for row in srows:
    if row['stage'] != 'measure_after':
        continue
    t = row['timestep']
    branch_outcomes = stroke_result.outcomes if row['branch'] == 'factual' else stroke_result.counterfactual_outcomes
    action = row['branch'] == 'factual' and t in stroke_result.interventions and pid in stroke_result.interventions[t].treated_indices
    event = branch_outcomes[t].events[pid]
    if action or event:
        print(stroke_path.name, row['trace_id'], 'week', t, row['branch'],
              'assigned', bool(action), 'event', float(event), row['facts']['patients'][str(pid)])
print('Exact engine equality and risk arithmetic: PASS')

Review persistent treatment multipliers, the direct before/after intervention rows, and both trajectories. A trace should explain what actually happened without inferring a pre-treatment risk by dividing the current risk by an assumed effect. Prediction scores and treatment indices reconcile with the normal engine run; fitting target performance requires a separate model verification check.

Trace files contain selected state facts plus compact output counts. Their `trace_id` references are unique **within each named run file**. These JSON files are local generated evidence; the notebook is reusable source.